# SMO — T4 Memory Benchmark (Colab/Kaggle)

Compara optimizadores sobre una T4 de 16 GB. **Sube este notebook a Colab,
elige Runtime > Change runtime type > T4 GPU, y ejecuta las celdas en orden.**

| Sección | Qué hace | Tiempo |
|---|---|---|
| Setup | clona repo + bitsandbytes | ~1 min |
| Sanity (ViT 3 ep) | valida el entorno end-to-end | ~12 min |
| **Killer demo** | ~700M params: espera AdamW=OOM, SMO-8bit lp=ok | ~20–30 min |
| H5/H7 (ViT 10 ep + SGD-M) | trayectorias + baseline SGD | ~60 min |
| H4 (permute_basis) | ¿depende de la localidad? | ~8 min |
| Análisis | resúmenes mean±std y loss-matched | segundos |

Resultados en `benchmarks/results/t4_*_memory_results*.json` (última celda: zip).


In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} ({p.total_memory / 1e9:.1f} GB)")
assert torch.cuda.is_available(), 'Activa GPU: Entorno de ejecucion > Cambiar tipo de entorno > T4'

In [ ]:
import os
BASE = "/content" if os.path.isdir("/content") else "/kaggle/working"

REPO_URL = "https://github.com/mcarbonell/supermario_optimizer.git"

%cd {BASE}
!rm -rf smo_optimizer
!git clone --depth 1 {REPO_URL} smo_optimizer
%cd smo_optimizer

%pip install -q bitsandbytes

!python -c "from smo import SMO, SMO8bit; import benchmarks; print('SMO import OK')"

In [ ]:
# ---- Sanity: TinyViT CIFAR-10, 3 epochs (~12 min) ----
!python -m benchmarks.suites.comparison.t4_memory_benchmark --suite vit --epochs 3 --amp --seed 1234 --tag colabsanity

## Killer demo v2 (~700M params)

Con fp32, AdamW necesita ~16 B/param (pesos+grads+m+v): ~11 GB estáticos + activaciones →
debería quedarse sin memoria en la T4. SMO-8bit con `--low_peak` (update por bandas, sin
tenedores a resolución completa) ronda 9 B/param → debería entrenar con holgura.
El script registra el OOM por optimizador sin abortar: la tabla final ES el resultado.


In [ ]:
# ---- Killer demo (~700M, ~20-30 min) ----
!python -m benchmarks.suites.comparison.t4_memory_benchmark --suite gpt \
    --d_model 1280 --layers 36 --block_size 256 --batch 8 --steps 200 \
    --amp --low_peak --eval_interval 50 --seed 1234 --tag big2

## H5/H7: ¿por qué gana en visión? Trayectorias + baseline SGD-M

Si SGD-M se acerca a SMO → el efecto es de-adaptivización (Adam suavizado ≈ territorio SGD).
Si no → la compresión aporta algo propio. Las trayectorias permiten el análisis
loss-matched (mejor test_acc a igual train loss).


In [ ]:
# ---- ViT 10 épocas con trayectorias + SGD-M (~60 min) ----
!python -m benchmarks.suites.comparison.t4_memory_benchmark --suite vit --epochs 10 --amp --seed 1234 --optimizers adamw,bnb8bit,sgdm,smo,smo8bit --tag hist

In [ ]:
# ---- H4: romper la localidad con permutación fija (~8 min) ----
# Predicción: si la ventaja muere aquí, el smoothing explota correlación entre
# coordenadas vecinas; si sobrevive, es dinámica de optimización pura.
!python -m benchmarks.suites.comparison.t4_memory_benchmark --suite vit --epochs 3 --amp --seed 1234 --optimizers smo8bit --permute_basis --tag perm

In [ ]:
# ---- Análisis: resumen multi-seed + comparación loss-matched ----
!python -m benchmarks.suites.comparison.t4_summarize
!python -m benchmarks.suites.comparison.t4_loss_matched --suite vit

In [ ]:
# ---- Tabla pandas de todo lo acumulado ----
import glob
import json

import pandas as pd

rows = []
for path in sorted(glob.glob("benchmarks/results/t4_*_memory_results*.json")):
    bundle = json.load(open(path))
    for r in bundle["runs"]:
        m = r["metrics"]
        rows.append({
            "bundle": path.name.replace("t4_", "").replace("_memory_results.json", ""),
            "optimizer": r["variant"],
            "status": m.get("status", "?"),
            "peak_alloc_MB": m.get("_peak_alloc_mb"),
            "state_MB": m.get("persistent_state_mb"),
            "metric": m.get(r.get("metric_key", "")),
            "coverage_pct": m.get("coverage_pct"),
        })

df = pd.DataFrame(rows)
with pd.option_context("display.max_columns", None, "display.width", 220, "display.max_rows", None):
    display(df)

In [ ]:
# ---- Descargar resultados ----
import shutil

shutil.make_archive("t4_results", "zip", "benchmarks/results")
try:
    from google.colab import files

    files.download("t4_results.zip")
except ImportError:
    print("En Kaggle: panel de Output -> t4_results.zip")